#### Aproximacion modelos numpyro

In [ ]:
#import dask.dataframe as dd
import pandas as pd
from sklearn.preprocessing import StandardScaler,OneHotEncoder
df = pd.read_parquet("../../../data/Train/train_final.parquet")
df.head()

In [ ]:
df = df[df.parado]

In [ ]:
# Inicializamos todo en 0
df['took_off'] = 0

# Para cada despegue, buscamos el máximo tiempo_esperado
antes_de_despegue = df.groupby('despegue')['tiempo_esperado'].max().reset_index()

# Hacemos merge para tener en df el máximo tiempo_esperado de su grupo
df = df.merge(antes_de_despegue, on='despegue', suffixes=('', '_max'))

# Condición: donde el tiempo_esperado es el máximo, marcamos took_off = 1
df.loc[df['tiempo_esperado'] == df['tiempo_esperado_max'], 'took_off'] = 1

# Si quieres, luego puedes eliminar la columna auxiliar
df = df.drop(columns=['tiempo_esperado_max'])

# Vemos el resultado
print(df['took_off'].value_counts())

#### Apartado A

In [ ]:
import pandas as pd
import numpy as np
import jax.numpy as jnp
import jax
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from sklearn.preprocessing import LabelEncoder

data = df.iloc[:100]

# --- 2. Preprocesamiento: codificamos las categóricas como índices
le_aircraft = LabelEncoder()
data['aircraft_type_idx'] = le_aircraft.fit_transform(data['aircraft_type'])

le_holding = LabelEncoder()
data['holding_point_idx'] = le_holding.fit_transform(data['holding_point'])

# --- 3. Creamos los arrays que usaremos en NumPyro
aircraft_type_idx = jnp.array(data['aircraft_type_idx'].values)
holding_point_idx = jnp.array(data['holding_point_idx'].values)
runway_occupied = jnp.array(data['runway_occupied'].values)
hold_pt_occupied = jnp.array(data['hold_pt_occupied'].values)
queue_length = jnp.array(data['queue_length'].values)
took_off = jnp.array(data['took_off'].values)

# --- 4. Definimos el modelo de regresión logística en NumPyro
def logistic_regression_model(aircraft_type_idx, holding_point_idx,
                               runway_occupied, hold_pt_occupied,
                               queue_length, took_off=None,
                               n_aircraft_types=None, n_holding_points=None):
    
    # NO calcular dentro del modelo: JAX necesita saberlo antes.
    aircraft_type_effect = numpyro.sample("aircraft_type_effect", dist.Normal(0., 1.).expand([n_aircraft_types]))
    holding_point_effect = numpyro.sample("holding_point_effect", dist.Normal(0., 1.).expand([n_holding_points]))
    
    beta_runway_occupied = numpyro.sample("beta_runway_occupied", dist.Normal(0., 1.))
    beta_hold_pt_occupied = numpyro.sample("beta_hold_pt_occupied", dist.Normal(0., 1.))
    beta_queue_length = numpyro.sample("beta_queue_length", dist.Normal(0., 1.))
    intercept = numpyro.sample("intercept", dist.Normal(0., 5.))
    
    logits = (intercept 
              + aircraft_type_effect[aircraft_type_idx]
              + holding_point_effect[holding_point_idx]
              + beta_runway_occupied * runway_occupied
              + beta_hold_pt_occupied * hold_pt_occupied
              + beta_queue_length * queue_length)
    
    with numpyro.plate("data", logits.shape[0]):
        numpyro.sample("obs", dist.Bernoulli(logits=logits), obs=took_off)


# --- 5. Corremos el modelo MCMC
nuts_kernel = NUTS(logistic_regression_model)
mcmc = MCMC(nuts_kernel, num_warmup=500, num_samples=1000, num_chains=1)
# Calculamos una vez afuera
n_aircraft_types = data['aircraft_type_idx'].nunique()
n_holding_points = data['holding_point_idx'].nunique()

# Ahora llamamos run() pasando los tamaños
mcmc.run(
    jax.random.PRNGKey(0),
    aircraft_type_idx=aircraft_type_idx,
    holding_point_idx=holding_point_idx,
    runway_occupied=runway_occupied,
    hold_pt_occupied=hold_pt_occupied,
    queue_length=queue_length,
    took_off=took_off,
    n_aircraft_types=n_aircraft_types,
    n_holding_points=n_holding_points
)

# --- 6. Vemos el resumen de la inferencia
mcmc.print_summary()

In [ ]:
# --- 7. Datos de test
data_test = df.dropna().iloc[500:1000].copy()

# Aplicamos los mismos LabelEncoders que ya teníamos entrenados
data_test['aircraft_type_idx'] = le_aircraft.transform(data_test['aircraft_type'])
data_test['holding_point_idx'] = le_holding.transform(data_test['holding_point'])

# Arrays para el test
aircraft_type_idx_test = jnp.array(data_test['aircraft_type_idx'].values)
holding_point_idx_test = jnp.array(data_test['holding_point_idx'].values)
runway_occupied_test = jnp.array(data_test['runway_occupied'].values)
hold_pt_occupied_test = jnp.array(data_test['hold_pt_occupied'].values)
queue_length_test = jnp.array(data_test['queue_length'].values)


# --- 8. Hacemos predicciones usando el posterior
from numpyro.infer import Predictive

predictive = Predictive(
    logistic_regression_model,
    posterior_samples=mcmc.get_samples(),  # usamos las muestras que aprendimos
    return_sites=["obs"]  # solo queremos la predicción de 'obs'
)

# Sampleamos (generamos predicciones)
preds = predictive(
    jax.random.PRNGKey(1),  # otra semilla
    aircraft_type_idx=aircraft_type_idx_test,
    holding_point_idx=holding_point_idx_test,
    runway_occupied=runway_occupied_test,
    hold_pt_occupied=hold_pt_occupied_test,
    queue_length=queue_length_test,
    took_off=None,  # porque ahora queremos predecir
    n_aircraft_types=n_aircraft_types,
    n_holding_points=n_holding_points
)

# --- 9. Convertimos las predicciones a clases
# preds['obs'] tiene forma (n_samples_mcmc, n_test)
# Tomamos el promedio sobre muestras → probabilidad media de took_off
probs_pred = preds['obs'].mean(axis=0)

# Si probabilidad > 0.5, predecimos 1 (despegó), si no 0
y_pred = (probs_pred > 0.5).astype(int)

# --- 10. Calculamos accuracy
# El y_real son los took_off reales de los test
y_real = jnp.array(data_test['took_off'].values)

accuracy = (y_pred == y_real).mean()

print(f"Accuracy en test: {accuracy:.3f}")

#### Apartado B

In [ ]:
import pandas as pd
import numpy as np
import jax.numpy as jnp
import jax
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from sklearn.preprocessing import LabelEncoder

data = df.iloc[:10000]
festivos_espana = [
    "2024-11-01",  # Todos los Santos (nacional)
    "2024-12-06",  # Día de la Constitución Española (nacional)
    "2024-12-08",  # Inmaculada Concepción (nacional)
    "2024-12-09",  # Traslado de la Inmaculada (cuando cae domingo)
    "2024-12-25",  # Navidad (nacional)
    "2024-12-26",  # San Esteban (festivo en Cataluña y otras)
    "2025-01-01",  # Año Nuevo (nacional)
    "2025-01-06",  # Reyes Magos / Epifanía del Señor (nacional)
    "2024-11-09",  # Virgen de la Almudena (local Madrid)
]

# --- 1. Prepara los datasets
mask_weekend = data["despegue"].dt.weekday > 5
mask_holiday = data["despegue"].dt.date.astype(str).isin(festivos_espana)
weekend_data = data[mask_weekend]
holiday_data = data[mask_holiday]
normal_data = data[(~mask_weekend) & (~mask_holiday)]

# Codificación categórica para cada dataset
for data in [weekend_data, holiday_data, normal_data]:
    data['aircraft_type_idx'] = LabelEncoder().fit_transform(data['aircraft_type'])
    data['holding_point_idx'] = LabelEncoder().fit_transform(data['holding_point'])

# --- 2. Define el modelo logístico básico
def logistic_regression_model(aircraft_type_idx, holding_point_idx,
                               runway_occupied, hold_pt_occupied,
                               queue_length, took_off=None,
                               n_aircraft_types=None, n_holding_points=None):
    
    aircraft_type_effect = numpyro.sample("aircraft_type_effect", dist.Normal(0., 1.).expand([n_aircraft_types]))
    holding_point_effect = numpyro.sample("holding_point_effect", dist.Normal(0., 1.).expand([n_holding_points]))
    beta_runway_occupied = numpyro.sample("beta_runway_occupied", dist.Normal(0., 1.))
    beta_hold_pt_occupied = numpyro.sample("beta_hold_pt_occupied", dist.Normal(0., 1.))
    beta_queue_length = numpyro.sample("beta_queue_length", dist.Normal(0., 1.))
    intercept = numpyro.sample("intercept", dist.Normal(0., 5.))
    
    logits = (intercept 
              + aircraft_type_effect[aircraft_type_idx]
              + holding_point_effect[holding_point_idx]
              + beta_runway_occupied * runway_occupied
              + beta_hold_pt_occupied * hold_pt_occupied
              + beta_queue_length * queue_length)
    
    with numpyro.plate("data", logits.shape[0]):
        numpyro.sample("obs", dist.Bernoulli(logits=logits), obs=took_off)

# --- 3. Función para correr MCMC
def fit_model(data):
    aircraft_type_idx = jnp.array(data['aircraft_type_idx'].values)
    holding_point_idx = jnp.array(data['holding_point_idx'].values)
    runway_occupied = jnp.array(data['runway_occupied'].values)
    hold_pt_occupied = jnp.array(data['hold_pt_occupied'].values)
    queue_length = jnp.array(data['queue_length'].values)
    took_off = jnp.array(data['took_off'].values)

    n_aircraft_types = data['aircraft_type_idx'].nunique()
    n_holding_points = data['holding_point_idx'].nunique()

    nuts_kernel = NUTS(logistic_regression_model)
    mcmc = MCMC(nuts_kernel, num_warmup=500, num_samples=1000, num_chains=1)
    
    mcmc.run(
        jax.random.PRNGKey(0),
        aircraft_type_idx=aircraft_type_idx,
        holding_point_idx=holding_point_idx,
        runway_occupied=runway_occupied,
        hold_pt_occupied=hold_pt_occupied,
        queue_length=queue_length,
        took_off=took_off,
        n_aircraft_types=n_aircraft_types,
        n_holding_points=n_holding_points
    )
    
    return mcmc

# --- 4. Entrena los tres modelos
mcmc_weekend = fit_model(weekend_data)
mcmc_holiday = fit_model(holiday_data)
mcmc_normal = fit_model(normal_data)


In [ ]:
import numpyro.diagnostics

def predict_model(mcmc, new_data):
    samples = mcmc.get_samples()
    
    # Extraemos parámetros
    intercept = samples['intercept'].mean()
    beta_runway_occupied = samples['beta_runway_occupied'].mean()
    beta_hold_pt_occupied = samples['beta_hold_pt_occupied'].mean()
    beta_queue_length = samples['beta_queue_length'].mean()
    aircraft_type_effect = samples['aircraft_type_effect'].mean(axis=0)
    holding_point_effect = samples['holding_point_effect'].mean(axis=0)

    # Armamos el logit para el nuevo caso
    logit = (intercept
             + aircraft_type_effect[new_data['aircraft_type_idx']]
             + holding_point_effect[new_data['holding_point_idx']]
             + beta_runway_occupied * new_data['runway_occupied']
             + beta_hold_pt_occupied * new_data['hold_pt_occupied']
             + beta_queue_length * new_data['queue_length'])
    
    prob = jax.nn.sigmoid(logit)
    return prob

# --- 5. Ejemplo: nuevo caso
new_case = {
    'aircraft_type_idx': 1,
    'holding_point_idx': 2,
    'runway_occupied': 0,
    'hold_pt_occupied': 1,
    'queue_length': 3
}

# Predicciones
p_weekend = predict_model(mcmc_weekend, new_case)
p_holiday = predict_model(mcmc_holiday, new_case)
p_normal = predict_model(mcmc_normal, new_case)

# Normalizamos para obtener la probabilidad de pertenencia
probs = jnp.array([p_weekend, p_holiday, p_normal])
probs_normalized = probs / probs.sum()

print(f"Probabilidades de pertenencia:")
print(f"Weekend: {probs_normalized[0]:.3f}")
print(f"Holiday: {probs_normalized[1]:.3f}")
print(f"Normal : {probs_normalized[2]:.3f}")
